In [1]:
#!pip install spectral-cube
#!pip install astroquery
#!pip install reproject
!pip install ccdproc
!pip install astropy pandas

In [18]:
import glob #importing glob library
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import numpy as np

import astropy
import astropy.units as u
from astropy.utils.data import download_file
from astropy.io import fits  # We use fits to open the actual data file
from astropy.utils import data
from astropy.wcs import wcs
from astropy.coordinates import SkyCoord
from astropy.time import Time
import pandas as pd
from astropy.nddata import CCDData

import ipywidgets as widgets
from IPython.display import display
import os
import ccdproc
import warnings
import copy

In [21]:
'Retrieving a set of flats with a binning factor of 2'
flat_fields = glob.glob('D:/Ay16Flats/*.fit')

In [22]:
'Silencing errors from ISO formatting of date in fits file metadata'
warnings.filterwarnings('ignore', category=Warning, append=True)

'Creating an array storing the paths of each flat in the library flat_fields'
ccd_list = [CCDData.read(f, unit="adu") for f in flat_fields]

'Finding the median value of each flat before taking the reciprocal to produce a scaling factor for flatfielding'
median_flats = []
for file_path in flat_fields:
    with fits.open(file_path) as hdul:
        flats_data = hdul[0].data
    median_fits = 1/np.median(flats_data)
    median_flats.append(median_fits)

In [5]:
'Producing a single flat composed of all flats in ccd_list'
master_flat = ccdproc.Combiner(ccd_list)
'Scaling the flat by the median value'
master_flat.scaling = median_flats
'For each point of data, take the median value of all flats at that point'
master_flat_data = master_flat.median_combine()

'Creating a master flat consisting of the steps'
fits_header = fits.Header(master_flat_data.header)
clean_hdu = fits.PrimaryHDU(data=master_flat_data.data, header=fits_header)
clean_hdu.writeto('mflat_512.fits', overwrite=True)

TypeError: can only concatenate tuple (not "NoneType") to tuple

In [6]:
fits_files = glob.glob('C:/Users/jules/OneDrive/Desktop/Research Test Files/*.fit')

In [7]:
'Creating paths to files both on a personal computer and external hard drive'
#mflat_2048 = 'C:/Users/Astroadmin/Documents/Ay16/EB NSVS01031772 Usable Files/2022 EB Files/03-22-2019/Flats_2022.0322/mflat.fits'
#mflat_512 = 'C:/Users/Astroadmin/mflat_512.fits'

raw_fits_files = glob.glob('C:/Users/jules/OneDrive/Desktop/Test Flat/*.fit')
mflat_2048 = 'C:/Users/jules/OneDrive/Desktop/Test Flat/mflat.fits'
mflat_512 = 'C:/Users/jules/mflat_512.fits'

In [8]:
'Pathing to the folder containing all usable fits files for the project'
#all_fits_files = glob.glob('D:/EB NSVS01031772 Usable Files/*.fit')

'Pathing to the folder containing all usable fits files for the project'

In [9]:
'Designated path is a variable which may be set to any path which is then called by the rest of the code'
#designated_path = 'D:/EB NSVS01031772 Usable Files/'
designated_path = 'C:/Users/jules/OneDrive/Desktop/Research Test Files/'

In [10]:
#hdul_test = fits.open('D:/EB NSVS01031772 Usable Files/2016 EB Files//03-30-2016/eb_R_90s_160329-059.fit')
#hdul_test[0].header

In [11]:
def file_by_date (directory):
    raw_fits_date = []
    fits_times = []

    for root, dirs, files in os.walk(directory):
        for filename in files:
            if filename.endswith(('.fits', '.fit')):
                file_path = os.path.join(root, filename)

                try:
                    with fits.open(file_path) as hdul:
                        date = hdul[0].header['DATE-OBS']
                        size = hdul[0].header['NAXIS1']

                    t = Time(date, format='isot', scale='utc')
                    MJD_fits = t.mjd
            
                    raw_fits_date.append({
                        'File Name': filename,
                        'Date': date,
                        'MJD': float(MJD_fits),
                        'Location': root,
                        'Size': size
                    })                        
            
                except (KeyError, OSError) as e: 
                    print(f"Skipping {filename}: {e}")

    raw_fits_date.sort(key=lambda x: x['Date'])

    return raw_fits_date

raw_fits_dict = file_by_date(designated_path)

In [12]:
def search_file(search_query, raw_fits_dict): #searchable dictionary for the flat for loop
    for i in raw_fits_dict:
        if i.get('File Name') == search_query:
            return os.path.join(i['Location'], i['File Name'])
        
    print(f"Error: Key '{search_query}' not found in the dictionary.")
    return None

search_file('test-0001.fit', raw_fits_dict)

Error: Key 'test-0001.fit' not found in the dictionary.


In [13]:
flatfield_select = widgets.Select(
    options=list(dict.fromkeys([i['Date'].split('T')[0] for i in raw_fits_dict])),
    description='Display:',
    disabled=False
)

In [19]:
warnings.filterwarnings('ignore', category=Warning, append=True)
master_flat_2048 = CCDData.read(mflat_2048, unit='adu', ignore_missing_simple=True)
master_flat_512 = CCDData.read(mflat_512, unit='adu', ignore_missing_simple=True)

SIZE_512 = 512
SIZE_2048 = 2048

def flatfield (dictionary, date):
    processed_data = []

    for i in dictionary:
        if i['Date'].split('T')[0] == date:
            raw_file_path = search_file(i['File Name'], raw_fits_dict)
            with fits.open(raw_file_path) as hdul:
                fits_reading = CCDData.read(raw_file_path, unit='adu', ignore_missing_simple=True)
                corrected_fits = copy.deepcopy(fits_reading)

                if i['Size'] == SIZE_512:
                    ccdproc.flat_correct(corrected_fits, flat=master_flat_512)
                
                elif i['Size'] == SIZE_2048:
                    ccdproc.flat_correct(corrected_fits, flat=master_flat_2048)

                else:
                    print(f'Error: File {i['File Name']} size incompatible')

                processed_data.append({
                    'metadata': i,
                    'image data': corrected_fits.data,
                    'header': corrected_fits.header
                })

            print(f"Processed {len(processed_data)} images for date: {date}")
            return processed_data

select = widgets.interactive(flatfield, dictionary=widgets.fixed(raw_fits_dict), date=flatfield_select)
display(select)

interactive(children=(Select(description='Display:', options=('2018-03-20',), value='2018-03-20'), Output()), …

In [20]:
def folder_by_date(processed_data):
    current_folder_date = ""
    prev_mjd = None
    folder_paths=[]

    for i in processed_data:
        item = i['metadata']
        mjd = item['MJD']
        flat_hdu = fits.PrimaryHDU(data=i['image data'], header=i['header'])
        
        if prev_mjd is None:
            current_folder_date = Time(mjd, format='mjd').iso.split()[0]
            
        else:
            if mjd - prev_mjd >= 0.104167:
                current_folder_date = Time(mjd, format='mjd').iso.split()[0]
                
        new_dir = f'C:/Users/Astroadmin/Desktop/EB_{current_folder_date}'
        name = f'EB_{current_folder_date}'
        
        current_path_info = {
            'Location': new_dir,
            'Name': name
        }
        
        os.makedirs(new_dir, exist_ok=True)

        if current_path_info not in folder_paths: 
            folder_paths.append(current_path_info)
        
        file_output = os.path.join(new_dir, item['File Name'])
        
        flat_hdu.writeto(file_output, overwrite=True)  
        
        prev_mjd = mjd

    return folder_paths

folder_options = folder_by_date(select.result)

NameError: name 'processed_data' is not defined

In [16]:
cube = np.zeros((len(raw_fits_dict),2048,2048))

In [ ]:
cube_file_select = widgets.Select(
    options=[i['Name'] for i in folder_options],
    description='Display:',
    disabled=False
)

In [ ]:
def cubed_normed_fits (directory):
    norm_fits = []
    
    for root, dirs, file in os.walk(f'C:/Users/Astroadmin/Desktop/{directory}'):
        for filename in file:
            if filename.endswith(('fit', 'fits')):
                full_path = os.path.join(root, filename)

                try:
                    with fits.open(full_path) as hdu_fits:
                        data_org = hdu_fits[0].data.astype(np.float32)
                        data_max = np.nanmax(data_org)
                        if data_max == 0: 
                            data_max = 1.0
                        data_norm = data_org / data_max
                        norm_fits.append(data_norm)

                except (KeyError, OSError) as e: 
                    print(f"Skipping {filename}: {e}")

    num_frames = len(norm_fits)
    item = select.result[0]['metadata']
    size = item['Size']
    cube = np.zeros((num_frames, size, size), dtype=np.float32)

    for i, data_array in enumerate(norm_fits):
                        cube[i,:,:] = data_array

    hdu_norm_new = fits.PrimaryHDU(cube)
    hdu_norm_new.writeto('cube_norm.fits', overwrite=True)

    cube_norm = fits.open('cube_norm.fits', memmap = False) 
    cube_norm_data = cube_norm[0].data

    return cube_norm_data

widgets.interactive(cubed_normed_fits, directory=cube_file_select)